# Baselines, Task Definition, and Ablations (Phase 7)

Runs in Google Colab. All real training and ablations happen here, never
on a laptop. Calls `evat.experiments.*` / `evat.models.*` — does not
reimplement the task, models, or training loop.

Task: see `docs/task_definition.md` — object category classification
from a temporal visual feature sequence, using YouTube-VOS's official
per-object category annotation as the label. Compares:

1. `TemporalMeanPoolBaseline` — non-temporal (mean-pool features -> MLP)
2. `TemporalGRUBaseline` — simple temporal (GRU -> masked mean pool -> MLP)
3. `VideoTransformer` — from-scratch Transformer (Phase 6, unmodified)

under IDENTICAL data split, features, sequence length, batch size,
optimizer, and training budget.

In [ ]:
%pip install -q -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import os
from pathlib import Path

os.environ["EVAT_DATA_ROOT"] = "/content/data"  # example only; set to the real path
dataset_root = Path(os.environ["EVAT_DATA_ROOT"]) / "youtube_vos"

## Build the task dataset (video-level split, per docs/task_definition.md)

In [ ]:
from evat.data.datasets.youtube_vos import build_video_index
from evat.experiments.config import Phase7ExperimentConfig
from evat.experiments.task_category import (
    build_category_label_map,
    build_category_samples,
    split_videos_by_video_id,
)
from evat.features.encoders import CNNEncoderConfig, CNNFeatureEncoder

config = Phase7ExperimentConfig.from_yaml("configs/phase7.yaml")

all_train_videos = build_video_index(dataset_root, split="train")
train_videos, val_videos = split_videos_by_video_id(
    all_train_videos, val_fraction=config.val_fraction, seed=config.seed
)
label_map = build_category_label_map(train_videos)
print("categories:", label_map)

encoder = CNNFeatureEncoder(CNNEncoderConfig(pretrained=True, feature_dim=None)).to(device)

train_samples = build_category_samples(
    train_videos,
    str(dataset_root),
    label_map,
    encoder,
    num_frames_per_video=config.num_frames_per_video,
    sequence_length=config.sequence_length,
    stride=config.stride,
)
val_samples = build_category_samples(
    val_videos,
    str(dataset_root),
    label_map,
    encoder,
    num_frames_per_video=config.num_frames_per_video,
    sequence_length=config.sequence_length,
    stride=config.stride,
)
print("train samples:", len(train_samples), "val samples:", len(val_samples))

## Controlled comparison: baseline MLP vs. GRU vs. Transformer

In [ ]:
import time

import torch

from evat.experiments.classifier_training import evaluate_classifier, train_classifier_step
from evat.experiments.record import save_experiment_result
from evat.models.temporal_baseline import TemporalMeanPoolBaseline
from evat.models.temporal_gru import TemporalGRUBaseline
from evat.models.transformer.config import TransformerConfig
from evat.models.transformer.model import VideoTransformer

feature_dim = train_samples[0].sequence.feature_dim
num_classes = len(label_map)


def to_batches(samples, batch_size):
    batches = []
    for i in range(0, len(samples), batch_size):
        chunk = samples[i : i + batch_size]
        features = torch.stack([torch.from_numpy(s.sequence.features) for s in chunk]).to(device)
        mask = torch.stack([torch.from_numpy(s.sequence.validity) for s in chunk]).to(device)
        labels = torch.tensor([s.label for s in chunk], device=device)
        batches.append((features, mask, labels))
    return batches


train_batches = to_batches(train_samples, config.batch_size)
val_batches = to_batches(val_samples, config.batch_size)

models = {
    "baseline_mlp": TemporalMeanPoolBaseline(feature_dim, hidden_dim=64, num_classes=num_classes),
    "temporal_gru": TemporalGRUBaseline(feature_dim, hidden_dim=64, num_classes=num_classes),
    "transformer": VideoTransformer(TransformerConfig.from_yaml("configs/transformer.yaml")),
}

results = {}
for name, model in models.items():
    torch.manual_seed(config.seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

    start = time.time()
    for _epoch in range(config.epochs):
        for features, mask, labels in train_batches:
            train_classifier_step(model, features, mask, labels, optimizer)
    runtime_seconds = time.time() - start

    metrics = evaluate_classifier(model, val_batches, num_classes)
    print(name, metrics, "runtime:", runtime_seconds)
    results[name] = metrics

    save_experiment_result(
        "results/phase7",
        name,
        config={"phase7": config.__dict__, "model": name},
        metrics=metrics,
        git_commit="<fill in from `git rev-parse HEAD`>",
        dataset_version="<fill in verified YouTube-VOS version/date>",
        hardware=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        runtime_seconds=runtime_seconds,
    )

## Ablations

One-variable-at-a-time, using the Transformer as the base configuration
from the comparison above.

1. **Sequence length** — rebuild `train_samples`/`val_samples` with
   `sequence_length` from `configs/phase7.yaml`'s `ablations.sequence_length`
   list, retrain, record accuracy/macro-F1/runtime per length.
2. **Positional encoding** — since `TransformerConfig` currently requires
   `positional_encoding == "sinusoidal"`, a no-positional-encoding variant
   means monkeypatching `SinusoidalPositionalEncoding.forward` to be a
   no-op for this ablation only (not a permanent architecture change).
3. **Masking** — compare the Transformer evaluated with the real validity
   mask vs. a mask forced to all-True (i.e. padded positions incorrectly
   treated as valid), to demonstrate padded positions would otherwise
   contaminate the representation. Only meaningful if the built samples
   actually contain padding (short videos / `sequence_length` >
   available frames).
4. **Model size** — a "small" vs. "medium" `TransformerConfig`
   (`d_model`/`num_layers`/`num_heads`/`d_ff`), same training budget.

Each ablation should call `save_experiment_result("results/phase7/ablations", ...)`
with a descriptive experiment name.

## Save results and analysis

Update `docs/experiments.md` with the actual printed metrics/runtime
above. Answer, using only measured results: does temporal modeling help?
does the Transformer beat the GRU? does positional encoding matter? does
sequence length matter? does masking matter? Report negative results if
that is what the numbers show.